# Debugging Flask Huffman Decompression Workflow

This notebook exercises the `/decompress` route and inspects response payloads, ZIP handling, subprocess output, and temp directory behavior.

## Section: Import Required Libraries

Import required Python modules for testing the Flask application and inspecting responses.

In [ ]:
import base64
import io
import json
import logging
import os
import pathlib
import tempfile
import traceback

from flask import Flask
from main import app, BASE_DIR, COMPRESSOR_DIR, DECOMPRESS_EXE, COMPRESS_EXE


: 

## Section: Load Flask App and Route Definitions

Verify the Flask app object and extract route information for `/decompress`.

In [ ]:
app.testing = True
client = app.test_client()

print('Flask app loaded.')
print('Base directory:', BASE_DIR)
print('Compressor directory:', COMPRESSOR_DIR)
print('Decompress binary:', DECOMPRESS_EXE)
print('Compress binary:', COMPRESS_EXE)


## Section: Reproduce Decompression Request for Video Archive

Construct a fake `.mp4` test file, compress it using the existing `/compress` route, and then submit the resulting archive to `/decompress`.

In [ ]:
from io import BytesIO

# Create a fake video-like binary payload with a known extension.
video_bytes = b'\x00\x01\x02\x03' * 1024 * 50  # ~200KB of binary data
video_name = 'test_video.mp4'

# Build multipart form data for compression.
data = {
    'files': (BytesIO(video_bytes), video_name)
}
response = client.post('/compress', data=data, content_type='multipart/form-data')
print('Compress response status:', response.status_code)
print('Compress response type:', response.content_type)
try:
    compress_json = response.get_json(force=False, silent=False)
    print('Compress response JSON keys:', list(compress_json.keys()))
except Exception as err:
    print('Compress JSON parse failed:', err)
    print(response.data.decode('utf-8', errors='replace'))
    raise

if compress_json and compress_json.get('results'):
    compressed_result = compress_json['results'][0]
    print('Compress result:', compressed_result)
    compressed_zip_b64 = compressed_result.get('data_b64')
    compressed_zip_name = compressed_result.get('compressed_filename')
    compressed_zip_bytes = base64.b64decode(compressed_zip_b64)
else:
    raise RuntimeError('Compression did not return expected result')

# Send decompression request with compressed zip.
zip_io = BytesIO(compressed_zip_bytes)
zip_io.name = compressed_zip_name

decompress_data = {
    'files': (zip_io, compressed_zip_name)
}
response = client.post('/decompress', data=decompress_data, content_type='multipart/form-data')
print('Decompress response status:', response.status_code)
print('Decompress response type:', response.content_type)
print('Decompress raw response head:', response.data[:200])
try:
    decompress_json = response.get_json(force=False, silent=False)
    print('Decompress response JSON keys:', list(decompress_json.keys()))
except Exception as err:
    print('Decompress JSON parse failed:', err)
    decoded = response.data.decode('utf-8', errors='replace')
    print('Raw response body:\n', decoded)
    raise

print('Decompress result:', decompress_json['results'][0])


## Section: Inspect Frontend Response and JSON Parsing

Diagnose whether invalid JSON is returned, or if the failure is from HTML or a backend exception before jsonify().

In [ ]:
def inspect_response(res):
    print('Status code:', res.status_code)
    print('Content-Type:', res.content_type)
    print('Raw body preview:', res.data[:200])
    try:
        payload = res.get_json(force=False, silent=False)
        print('Parsed JSON payload:', json.dumps(payload, indent=2)[:1000])
        return payload
    except Exception as err:
        print('JSON parse error:', err)
        try:
            print('Raw text body:', res.data.decode('utf-8', errors='replace')[:2000])
        except Exception as inner:
            print('Unable to decode raw body:', inner)
        return None

# Use the last decompression response if available.
try:
    inspect_response(response)
except NameError:
    print('Response object not available in this notebook cell')


## Section: Audit Decompress Workflow in Flask Route

Inspect the `/decompress` route implementation and extract the relevant helper function logic.

In [ ]:
import inspect

print(inspect.getsource(app.view_functions['decompress']))


## Section: Add Debug Logging Around Subprocess and Temp Paths

Review the helper functions and confirm the logging we added around ZIP extraction, subprocess execution, and temp file resolution.

In [ ]:
print(inspect.getsource(app.view_functions['decompress']))
print('\n---\n')
print(inspect.getsource(extract_compressed_bin_from_zip))
print('\n---\n')
print(inspect.getsource(run_compressor))


## Section: Verify Output File Existence and Temp Directory Contents

Decode the returned archive and inspect the inner file names produced by the decompressor.

In [ ]:
import zipfile

if 'decompress_json' in globals() and decompress_json is not None:
    results = decompress_json.get('results', [])
    if results:
        result = results[0]
        print('Decompress result summary:', result)
        if result.get('data_b64'):
            output_zip_bytes = base64.b64decode(result['data_b64'])
            with zipfile.ZipFile(io.BytesIO(output_zip_bytes), 'r') as zf:
                print('Output ZIP members:', zf.namelist())
                for info in zf.infolist():
                    if info.filename != 'metadata.json':
                        with zf.open(info) as member:
                            print('Member name:', info.filename, 'size:', info.file_size)
    else:
        print('No results in decompress response')
else:
    print('No decompress_json available in workspace')


## Section: Analyze ZIP Extraction and Media-Specific Failure Cases

Check how ZIP extraction handles a compressed `.bin` archive and whether file naming or extension resolution is correct for video-like files.

In [ ]:
from main import extract_compressed_bin_from_zip, save_uploaded_file

# If we have a compressed zip generated earlier, inspect the extracted binary.
if 'compressed_zip_bytes' in globals():
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_path = pathlib.Path(tmpdir)
        zip_path = tmp_path / compressed_zip_name
        zip_path.write_bytes(compressed_zip_bytes)
        extracted = extract_compressed_bin_from_zip(zip_path, tmp_path)
        print('ZIP extraction returned:', extracted)
        if extracted:
            print('Extracted file exists:', extracted.exists())
            print('Extracted filename:', extracted.name)
            print('Extracted file size:', extracted.stat().st_size)
else:
    print('No compressed_zip_bytes available to inspect ZIP extraction')


## Section: Document Reproducible Issues for GitHub Tracking

Summarize the decompression findings in issue format after reviewing the route behavior and test outputs.